In [1]:
"""
==================================================================================
 CUSTOMER RETENTION INTELLIGENCE PLATFORM - SECONDARY DATASET (Olist)
 Phase 1: Data Understanding -> File 3 of 5: olist_order_items_dataset.csv
==================================================================================
 Purpose : Explore order line-item data (used for Tableau dashboard).
           Contains one row per product per order, with price and freight.
==================================================================================
"""

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)


def section(title: str) -> None:
    print("\n" + "=" * 90)
    print(f" {title}")
    print("=" * 90)


# ----------------------------------------------------------------------------------
# 1. DATA COLLECTION
# ----------------------------------------------------------------------------------
section("1. DATA COLLECTION")

DATA_PATH = "/Users/meetnakrani/Library/Mobile Documents/com~apple~CloudDocs/Customer-Retention-intelligence-Platform/DataSet/RAW/secondary Data/olist_order_items_dataset.csv"
df = pd.read_csv(DATA_PATH, parse_dates=["shipping_limit_date"])

print(f"Dataset loaded successfully from: {DATA_PATH}")
print(f"Shape of dataset -> Rows: {df.shape[0]}, Columns: {df.shape[1]}")


# ----------------------------------------------------------------------------------
# 2. DATASET STRUCTURE
# ----------------------------------------------------------------------------------
section("2. DATASET STRUCTURE (First & Last Records)")
print("\n--- First 5 records ---")
print(df.head())
print("\n--- Last 5 records ---")
print(df.tail())

section("2.1 COLUMN NAMES & DATA TYPES")
print(df.dtypes)

section("2.2 MEMORY USAGE & GENERAL INFO")
df.info(memory_usage="deep")


# ----------------------------------------------------------------------------------
# 3. MISSING VALUE ANALYSIS
# ----------------------------------------------------------------------------------
section("3. MISSING VALUE ANALYSIS")

missing_count = df.isnull().sum()
missing_percent = (missing_count / len(df)) * 100
missing_summary = pd.DataFrame({
    "Missing Count": missing_count,
    "Missing %": missing_percent.round(2)
})
missing_summary = missing_summary[missing_summary["Missing Count"] > 0].sort_values("Missing %", ascending=False)

if missing_summary.empty:
    print("No missing values found in the dataset.")
else:
    print(missing_summary)


# ----------------------------------------------------------------------------------
# 4. DUPLICATE RECORD CHECK
# ----------------------------------------------------------------------------------
section("4. DUPLICATE RECORD CHECK")
print(f"Fully duplicated rows                       : {df.duplicated().sum()}")
print(f"Duplicated (order_id + order_item_id) combos: "
      f"{df.duplicated(subset=['order_id', 'order_item_id']).sum()}")


# ----------------------------------------------------------------------------------
# 5. STATISTICAL SUMMARY - PRICE & FREIGHT
# ----------------------------------------------------------------------------------
section("5. STATISTICAL SUMMARY - PRICE & FREIGHT VALUE")
print(df[["price", "freight_value"]].describe().T)

section("5.1 ITEMS PER ORDER")
items_per_order = df.groupby("order_id")["order_item_id"].count()
print(items_per_order.describe())
print(f"\nOrders with more than 1 item: {(items_per_order > 1).sum()} "
      f"({round((items_per_order > 1).mean() * 100, 2)}% of all orders)")


# ----------------------------------------------------------------------------------
# 6. SELLER & PRODUCT CARDINALITY
# ----------------------------------------------------------------------------------
section("6. SELLER & PRODUCT CARDINALITY")
print(f"Unique orders     : {df['order_id'].nunique()}")
print(f"Unique products   : {df['product_id'].nunique()}")
print(f"Unique sellers    : {df['seller_id'].nunique()}")

print("\nTop 10 sellers by number of items sold:")
print(df["seller_id"].value_counts().head(10))


# ----------------------------------------------------------------------------------
# 7. SHIPPING LIMIT DATE RANGE
# ----------------------------------------------------------------------------------
section("7. SHIPPING LIMIT DATE RANGE")
print(f"Earliest shipping_limit_date: {df['shipping_limit_date'].min()}")
print(f"Latest shipping_limit_date  : {df['shipping_limit_date'].max()}")


# ----------------------------------------------------------------------------------
# 8. OUTLIER GLANCE (PRICE & FREIGHT)
# ----------------------------------------------------------------------------------
section("8. OUTLIER GLANCE - PRICE & FREIGHT")
for col in ["price", "freight_value"]:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    upper_bound = q3 + 1.5 * iqr
    outliers = (df[col] > upper_bound).sum()
    print(f"{col}: IQR upper bound = {round(upper_bound, 2)} -> "
          f"{outliers} potential high-end outliers ({round(outliers / len(df) * 100, 2)}%)")


# ----------------------------------------------------------------------------------
# 9. INITIAL DATA QUALITY OBSERVATIONS
# ----------------------------------------------------------------------------------
section("9. INITIAL DATA QUALITY OBSERVATIONS")

observations = f"""
1. Dataset contains {df.shape[0]} order-line records covering {df['order_id'].nunique()}
   unique orders, {df['product_id'].nunique()} products and {df['seller_id'].nunique()} sellers.
2. Grain of this table is one row per (order_id, order_item_id) -> this is the
   central fact table for the Tableau dashboard; it joins to orders (order_id),
   products (product_id), and an implied sellers table (seller_id).
3. No missing values or duplicate line items were found.
4. 'price' and 'freight_value' both show right-skewed distributions with a
   meaningful number of high-end outliers -> consider capping/binning for
   cleaner Tableau visuals (e.g. box plots, histograms).
5. A notable share of orders contain more than one item -> useful for building
   an "average items per order" KPI on the dashboard.
"""
print(observations)

section("ORDER ITEMS DATASET - DATA UNDERSTANDING COMPLETE")


 1. DATA COLLECTION
Dataset loaded successfully from: /Users/meetnakrani/Library/Mobile Documents/com~apple~CloudDocs/Customer-Retention-intelligence-Platform/DataSet/RAW/secondary Data/olist_order_items_dataset.csv
Shape of dataset -> Rows: 112650, Columns: 7

 2. DATASET STRUCTURE (First & Last Records)

--- First 5 records ---
                           order_id  order_item_id                        product_id                         seller_id shipping_limit_date   price  \
0  00010242fe8c5a6d1ba2dd792cb16214              1  4244733e06e7ecb4970a6e2683c13e61  48436dade18ac8b2bce089ec2a041202 2017-09-19 09:45:35   58.90   
1  00018f77f2f0320c557190d7a144bdd3              1  e5f2d52b802189ee658865ca93d83a8f  dd7ddc04e1b6c2c614352b383efe2d36 2017-05-03 11:05:13  239.90   
2  000229ec398224ef6ca0657da4fc703e              1  c777355d18b72b67abbeef9df44fd0fd  5b51032eddd242adc84c38acab88f23d 2018-01-18 14:48:30  199.00   
3  00024acbcdf0a6daa1e931b038114c75              1  7634da152a4610f